In [5]:
import pandas as pd
import numpy as np
import fastparquet

In [6]:
# Arquivos de Populacao

pop10 = pd.read_excel('Dados-apoio/proj-2015-2019-POP-0a10anos-RIPSA.xlsx')
pop12 = pd.read_excel('Dados-apoio/proj-2015-2019-POP-0a12anos-RIPSA.xlsx')
pop11a59 = pd.read_excel('Dados-apoio/proj-2015-2019-POP-11a59-RIPSA.xlsx')
pop60 = pd.read_excel('Dados-apoio/proj-2015-2019-POP-60+RIPSA.xlsx')
pop_geral = pd.read_excel('Dados-apoio/proj-2015-2019-POP-GERAL-RIPSA.xlsx')

pops = [pop12,pop11a59,pop60,pop_geral]

In [7]:
####################################################################################################################
# Carrega os bancos Basico (com distancias e tempos), municipio por CIR e RAS e Sinan
base01 = pd.read_excel('Dados-iniciais/base01.xlsx')
muni_cir = pd.read_excel('Dados-iniciais/_Muni_por_Macro_DRS_CIR.xlsx')
sinan = pd.read_parquet('/home/usuario/Documentos/Lucas/Projetos estudos/Escorpiao_pesa/Dados-processados/2_df_recodificado.parquet', 
                        filters=[('ANO', 'in', list(range(2015, 2020)))] # Com filtro interno
)

# Sinan para construir variavel MG
sinan_mg = sinan

# Padroniza codigo municipio do Sinan como numero inteiro
sinan['ID_MN_RESI'] = sinan['ID_MN_RESI'].astype(int)

# Converter ampolas para número
for col in ["NU_AMPOL_8", "NU_AMPOL_9"]:
    sinan_mg[col] = pd.to_numeric(sinan_mg[col], errors="coerce").fillna(0)





####################################################################################################################
# Cria variavel Moderado/Grave (MG)


# =========================================================
# FILTRO DE CASOS MODERADOS/GRAVES QUALIFICADOS
# ESCORPIONISMO
# =========================================================


# ---------------------------------------------------------
# CRITÉRIOS FORTES
# Isoladamente já sugerem fortemente MG
# ---------------------------------------------------------

criterio_forte = (

    # SAA >= 2 ampolas
    (sinan_mg["NU_AMPOL_8"] >= 2) |

    # SAEsc >= 2 ampolas
    (sinan_mg["NU_AMPOL_9"] >= 2) |

    # Óbito por animais peçonhentos
    (sinan_mg["EVOLUCAO"] == "Obito por ap") |

    # Manifestações vagais
    (sinan_mg["CLI_VAGAIS"] == "Sim")

)

# ---------------------------------------------------------
# CRITÉRIOS ASSOCIATIVOS
# Variáveis sujeitas a erro de preenchimento,
# mas que em conjunto aumentam a probabilidade
# de representar MG
# ---------------------------------------------------------

criterio_associativo = (

    # Soroterapia + classificação moderado/grave
    (
        (sinan_mg["CON_SOROTE"] == "Sim") &
        (sinan_mg["TRA_CLASSI"].isin(["Moderado", "Grave"]))
    ) |

    # Soroterapia + manifestações sistêmicas
    (
        (sinan_mg["CON_SOROTE"] == "Sim") &
        (sinan_mg["MCLI_SIST"] == "Sim")
    ) |

    # Soroterapia + complicações sistêmicas
    (
        (sinan_mg["CON_SOROTE"] == "Sim") &
        (sinan_mg["COM_SISTEM"] == "Sim")
    )

)

# ---------------------------------------------------------
# FILTRO FINAL
# ---------------------------------------------------------

filtro_mg = criterio_forte | criterio_associativo


# ---------------------------------------------------------
# CRIAR VARIÁVEL BINÁRIA (OPCIONAL)
# ---------------------------------------------------------

sinan_mg["MG"] = np.where(filtro_mg, 1, 0)

# ---------------------------------------------------------
# CONFERÊNCIA
# ---------------------------------------------------------

print("Total de casos:", len(sinan_mg))
print("Moderados/Graves qualificados:", filtro_mg.sum())
print("Proporção:", round(filtro_mg.mean() * 100, 2), "%")


# Recodificando valores
#sinan_mg.loc[sinan_mg['TOTAL_MG'] == False, 'TOTAL_MG'] = 'Leve'
#sinan_mg.loc[sinan_mg['TOTAL_MG'] == True, 'TOTAL_MG'] = 'MG'

Total de casos: 117357
Moderados/Graves qualificados: 4475
Proporção: 3.81 %


In [ ]:
# Junta todos os arquivos de populacao

pop_merge = pop10.copy() # Cria uma cópia para não mexer no original

for i in pops:
    # O merge traz as colunas novas e você salva o resultado em pop_merge
    pop_merge = pop_merge.merge(
        right=i.iloc[:, [0, 2]], 
        how='left', 
        on='IBGE'
    )

In [ ]:
# Junta o banco de populacoes com a base 01

df = (
    base01
    .drop_duplicates()
    .merge(
        pop_merge.drop_duplicates(),
        left_on="MUNI_REFERENCIADO",
        right_on="MUNI_NOME",
        how="left",
        indicator=True
    )
)

In [ ]:
# Junta df com banco de regioes
df = (
    df.merge(
    right=muni_cir,
    how='left',
    left_on="MUNI_NOME",
    right_on="MUNI_NOME",
    indicator='merge_flag'
).copy()
)

In [ ]:
# Cria coluna de total de casos (sem distincao por faixa etaria)
total_casos = sinan_mg['ID_MN_RESI'].value_counts().reset_index(name='TOTAL_CASOS')

In [ ]:
sinan_mg.columns

Index(['DT_SIN_PRI', 'SEM_PRI', 'ANO_NASC', 'NU_IDADE_N', 'CS_SEXO',
       'CS_GESTANT', 'CS_RACA', 'CS_ESCOL_N', 'ID_MN_RESI', 'ID_OCUPA_N',
       'ANT_DT_ACI', 'ANT_UF', 'ANT_MUNIC_', 'SG_UF', 'ANT_TEMPO_',
       'ANT_LOCA_1', 'MCLI_LOCAL', 'CLI_DOR', 'CLI_EDEMA', 'CLI_EQUIMO',
       'CLI_NECROS', 'CLI_LOCAL_', 'CLI_LOCA_1', 'MCLI_SIST', 'CLI_NEURO',
       'CLI_HEMORR', 'CLI_VAGAIS', 'CLI_MIOLIT', 'CLI_RENAL', 'CLI_OUTR_2',
       'CLI_OUTR_3', 'CLI_TEMPO_', 'TP_ACIDENT', 'ANI_TIPO_1', 'ANI_SERPEN',
       'ANI_ARANHA', 'ANI_LAGART', 'TRA_CLASSI', 'CON_SOROTE', 'NU_AMPOLAS',
       'NU_AMPOL_1', 'NU_AMPOL_8', 'NU_AMPOL_6', 'NU_AMPOL_4', 'NU_AMPO_7',
       'NU_AMPO_5', 'NU_AMPOL_9', 'NU_AMPOL_3', 'COM_LOC', 'COM_SECUND',
       'COM_NECROS', 'COM_COMPOR', 'COM_DEFICT', 'COM_APUTAC', 'COM_SISTEM',
       'COM_RENAL', 'COM_EDEMA', 'COM_SEPTIC', 'COM_CHOQUE', 'DOENCA_TRA',
       'EVOLUCAO', 'DT_OBITO', 'DT_ENCERRA', 'DT_DIGITA', 'IDADE_TIPO',
       'IDADE_COMPLETA', 'IDADE_ANOS',

In [ ]:
# Junta total de casos ao banco base
df = df.merge(
    right=total_casos,
    right_on='ID_MN_RESI',
    left_on='IBGE',
    how='left'
    )


In [ ]:
df.to_excel('Dados-iniciais/base02.xlsx')

In [ ]:
'''
# Codigo comparacao

comparacao = (
    base01[["MUNI_REFERENCIADO"]]
    .drop_duplicates()
    .merge(
        pop_merge[["MUNI_NOME"]].drop_duplicates(),
        left_on="MUNI_REFERENCIADO",
        right_on="MUNI_NOME",
        how="left",
        indicator=True
    )
)

nao_encontrados = comparacao[comparacao["_merge"] == "left_only"]

print(nao_encontrados)
'''

'\n# Codigo comparacao\n\ncomparacao = (\n    base01[["MUNI_REFERENCIADO"]]\n    .drop_duplicates()\n    .merge(\n        pop_merge[["MUNI_NOME"]].drop_duplicates(),\n        left_on="MUNI_REFERENCIADO",\n        right_on="MUNI_NOME",\n        how="left",\n        indicator=True\n    )\n)\n\nnao_encontrados = comparacao[comparacao["_merge"] == "left_only"]\n\nprint(nao_encontrados)\n'